In [1]:
from reedsolo import RSCodec, rs_calc_syndromes
import os
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt

import sys
from pathlib import Path

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

from rs.channels import qsc_erasure_channel
from rs.dataset_gen import bytes_to_bits, get_zero_mask, RSPositionDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

from torchvision.ops import sigmoid_focal_loss
from rs.decoder import HybridDecoder


Device: cpu


In [2]:
class PositionPredictor(nn.Module):
    def __init__(self, use_batchnorm=False, dropout_rate=0.0):
        super().__init__()

        layers = []

        layers.append(nn.Linear(511, 512))
        if use_batchnorm:
            layers.append(nn.BatchNorm1d(512))
        layers.append(nn.ReLU())
        if dropout_rate > 0:
            layers.append(nn.Dropout(dropout_rate))
        
        for _ in range(3):
            layers.append(nn.Linear(512, 512))
            if use_batchnorm:
                layers.append(nn.BatchNorm1d(512))
            layers.append(nn.ReLU())
            if dropout_rate > 0:
                layers.append(nn.Dropout(dropout_rate))
        
        layers.append(nn.Linear(512, 255))

        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

In [23]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=1.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    
    def forward(self, inputs, targets):
        return sigmoid_focal_loss(inputs, targets, alpha=self.alpha, gamma=self.gamma, reduction='mean')
    
def get_criterion(loss_type):
    if loss_type == 'bce':
        return nn.BCEWithLogitsLoss()
    if loss_type == 'bce_weight':
        return nn.BCEWithLogitsLoss(pos_weight=torch.tensor([3.0]).to(device))
    elif loss_type == 'focal':
        return FocalLoss()

In [16]:
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0
    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, p_err, p_erase, num_samples=1000):
    model.eval()
    rsc = RSCodec(32)
    hybrid = HybridDecoder(model)

    classic_success = 0
    hybrid_success = 0
    classic_hint_success = 0

    for _ in range(num_samples):
        msg = os.urandom(223)
        codeword = rsc.encode(msg)
        noisy, erasure_pos = qsc_erasure_channel(codeword, p_err, p_erase)

        try:
            decoded, _, _ = rsc.decode(noisy)
            if bytes(decoded) == msg:
                classic_success += 1
        except:
            pass

        try:
            decoded, _, _ = rsc.decode(noisy, erase_pos=erasure_pos)
            if bytes(decoded) == msg:
                classic_hint_success += 1
        except:
            pass

        decoded = hybrid.decode(noisy, device)
        if decoded == msg:
            hybrid_success += 1
    
    return {
        'classic': classic_success / num_samples,
        'hybrid': hybrid_success / num_samples,
        'classic_hint': classic_hint_success / num_samples
    }

In [ ]:
def run_experiment(config, dataset, epochs=30):
    loader = DataLoader(dataset, batch_size=256, shuffle=True)

    model = PositionPredictor(
        use_batchnorm=config.get('batchnorm', False),
        dropout_rate=config.get('dropout', 0.0)
    ).to(device)

    criterion = get_criterion(config.get('loss', 'bce'))
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    for epoch in range(epochs):
        loss = train_epoch(model, loader, criterion, optimizer)
    
    result = evaluate(model, P_ERR, P_ERASE_TEST)
    result['config'] = config
    result['final_loss'] = loss

    return result, model

In [6]:
P_ERR = 0.02
P_ERASE_TRAIN = 0.06
P_ERASE_TEST = 0.06
TRAIN_SIZE = 50000

print("Generating dataset...")
dataset = RSPositionDataset(TRAIN_SIZE, P_ERR, P_ERASE_TRAIN)

Generating dataset...


In [ ]:
print("=== Comparing Loss functions ===")
print(f'{"Loss":<15} {"Hybrid":<10} {"Classic":<10}')
print("-" * 35)

loss_results = []
for loss_type in ['bce', 'bce_weight', 'focal']:
    config = {'loss' : loss_type}
    result, _ = run_experiment(config, dataset)
    loss_results.append(result)
    print(f'{loss_type:<15} {result["hybrid"]:<10.3f} {result["classic"]:<10.3f}')

=== Comparing Loss functions ===
Loss            Hybrid     Classic   
-----------------------------------
bce             0.646      0.179     
bce_weight      0.144      0.203     
focal           0.692      0.188     


In [18]:
best_loss = max(loss_results, key=lambda x: x['hybrid'])['config']['loss']
print(f'Best loss: {best_loss}')

Best loss: bce


In [19]:
print("=== Tuning BCE weight ===")
for w in [1.5, 2.0, 2.5, 3.0]:
    model = PositionPredictor().to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([w]).to(device))
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loader = DataLoader(dataset, batch_size=256, shuffle=True)
    
    for _ in range(30):
        train_epoch(model, loader, criterion, optimizer)
    
    result = evaluate(model, P_ERR, P_ERASE_TEST)
    
    model.eval()
    x = torch.tensor(dataset[0][0]).unsqueeze(0).to(device)
    with torch.no_grad():
        preds = (torch.sigmoid(model(x)) > 0.3).sum().item()
    
    print(f"weight={w:<4} Hybrid={result['hybrid']:.3f} Pred={preds}")

=== Tuning BCE weight ===
weight=1.5  Hybrid=0.534 Pred=33
weight=2.0  Hybrid=0.359 Pred=38
weight=2.5  Hybrid=0.276 Pred=46
weight=3.0  Hybrid=0.129 Pred=55


In [21]:
print("=== Tuning BCE weight (fine) ===")
for w in [1.0, 1.1, 1.2, 1.3, 1.4, 1.5]:
    model = PositionPredictor().to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([w]).to(device))
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loader = DataLoader(dataset, batch_size=256, shuffle=True)
    
    for _ in range(30):
        train_epoch(model, loader, criterion, optimizer)
    
    result = evaluate(model, P_ERR, P_ERASE_TEST)
    
    model.eval()
    x = torch.tensor(dataset[0][0]).unsqueeze(0).to(device)
    with torch.no_grad():
        preds = (torch.sigmoid(model(x)) > 0.3).sum().item()
    
    print(f"weight={w:<4} Hybrid={result['hybrid']:.3f} Pred={preds}")

=== Tuning BCE weight (fine) ===
weight=1.0  Hybrid=0.637 Pred=20
weight=1.1  Hybrid=0.582 Pred=30
weight=1.2  Hybrid=0.576 Pred=35
weight=1.3  Hybrid=0.605 Pred=32
weight=1.4  Hybrid=0.549 Pred=29
weight=1.5  Hybrid=0.460 Pred=29


In [22]:
print("\n=== Tuning Focal Loss ===")
for alpha in [0.1, 0.15, 0.2, 0.25]:
    for gamma in [1.0, 2.0]:
        model = PositionPredictor().to(device)
        criterion = FocalLoss(alpha=alpha, gamma=gamma)
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
        loader = DataLoader(dataset, batch_size=256, shuffle=True)
        
        for _ in range(30):
            train_epoch(model, loader, criterion, optimizer)
        
        result = evaluate(model, P_ERR, P_ERASE_TEST)
        
        model.eval()
        x = torch.tensor(dataset[0][0]).unsqueeze(0).to(device)
        with torch.no_grad():
            preds = (torch.sigmoid(model(x)) > 0.3).sum().item()
        
        print(f"α={alpha:<4} γ={gamma:<3} Hybrid={result['hybrid']:.3f} Pred={preds}")


=== Tuning Focal Loss ===
α=0.1  γ=1.0 Hybrid=0.580 Pred=12
α=0.1  γ=2.0 Hybrid=0.698 Pred=21
α=0.15 γ=1.0 Hybrid=0.659 Pred=15
α=0.15 γ=2.0 Hybrid=0.689 Pred=23
α=0.2  γ=1.0 Hybrid=0.700 Pred=19
α=0.2  γ=2.0 Hybrid=0.652 Pred=28
α=0.25 γ=1.0 Hybrid=0.706 Pred=23
α=0.25 γ=2.0 Hybrid=0.576 Pred=30
